# Anayaa.AI Submission Evidence Map

**Agents for Good**. Anayaa helps people navigate moral and life dilemmas with local-first, scripture-grounded, audited AI guidance.
Anayaa.AI is an Agents for Good submission because it applies a local-first, scripture-grounded multi-agent workflow to morally sensitive personal dilemmas. The system separates responsibilities across planning, retrieval, synthesis, audit, and finalization, then uses MCP as a narrow retrieval boundary so scripture access is tool-mediated instead of hidden inside an unrestricted prompt. Security and privacy are first-class product behavior: unsafe inputs are blocked before agent execution, personal identifiers are scrubbed before model-facing processing, and generated guidance must pass grounding and harmlessness checks before being shown. Deployability is demonstrated through a simple local beta path built around the `anayaa` CLI: setup once, serve locally, diagnose with doctor, and clean up explicitly.

## Key Concept Map

| Key concept | Where to demonstrate | Evidence anchors |
|---|---|---|
| Agent / multi-agent system (ADK) | Code | `backend/app/agents/adk_workflow.py`: ADK `Workflow`, `Runner`, named nodes for planner, optimizer, ReAct retry, retriever, synthesizer, judge, and finalize. |
| MCP Server | Code | `backend/app/mcp/milvus_retrieval_server.py`: FastMCP stdio server exposing `milvus_hybrid_search`, `graph_expand`, and `rerank_candidates_tool`; `backend/app/mcp/client.py`: allowlisted persistent MCP client. |
| Antigravity | Video | Show the project open in Antigravity, the local CLI flow, and the app responding to the demo query. Treat this as workflow/tooling evidence, not repo-code evidence. |
| Security features | Code or Video | `backend/app/api/routes/query.py`, `backend/app/security/firewall.py`, `backend/app/security/privacy_scrubber.py`, and `backend/app/observability/g_eval_judge.py` show pre-agent sanitization, prompt-injection blocking, PII scrubbing, rate limiting, and audit scoring. |
| Deployability | Video | Show the public beta path: installer or checkout, then `anayaa setup`, `anayaa serve`, and `anayaa doctor` / `anayaa release-check` if time allows. Code anchors are `scripts/install-anayaa.sh` and `scripts/anayaa`. |
| Agent skills / Agents CLI | Code or Video | |

In [19]:
from pathlib import Path

def is_repo_root(path):
    return (path / "README.md").exists() and (path / "backend").exists()

def find_repo_root(start=None):
    path = Path(start or Path.cwd()).resolve()
    candidates = [path, *path.parents]

    # Kaggle usually runs notebooks from /kaggle/working even after cloning into
    # /kaggle/working/Anayaa.AI, so also check common child directories.
    for base in [path, Path("/kaggle/working"), Path("/kaggle/input")]:
        if base.exists():
            candidates.extend([p for p in base.iterdir() if p.is_dir()])

    for candidate in candidates:
        if is_repo_root(candidate):
            return candidate.resolve()

    input_dir = Path("/kaggle/input")
    if input_dir.exists():
        for readme in input_dir.rglob("README.md"):
            candidate = readme.parent
            if is_repo_root(candidate):
                return candidate.resolve()

    raise RuntimeError("Could not find Anayaa.AI. In Kaggle, clone to /kaggle/working/Anayaa.AI or attach the repo as a dataset.")

ROOT = find_repo_root()
print(ROOT)

/Users/lakshmi/Desktop/Coding_challenge/Capstone/Anayaa.AI


In [20]:
def show(path, start, end):
    file_path = ROOT / path
    lines = file_path.read_text(encoding="utf-8").splitlines()
    print(f"\n# {path}:{start}-{end}\n")
    for line_no in range(start, min(end, len(lines)) + 1):
        print(f"{line_no:4}: {lines[line_no - 1]}")

## 1. Agent / Multi-Agent System (ADK)

 Anayaa is not a single prompt wrapper. The runtime uses an ADK workflow with separate responsibilities: query rewriting, optimization, planning, ReAct retry decisioning, MCP retrieval, synthesis, independent audit, and finalization.

In [21]:
show("backend/app/agents/adk_workflow.py", 11, 18)
show("backend/app/agents/adk_workflow.py", 476, 514)
show("backend/app/agents/adk_workflow.py", 623, 674)
show("backend/app/agents/adk_workflow.py", 739, 823)
show("backend/app/agents/adk_workflow.py", 1138, 1164)


# backend/app/agents/adk_workflow.py:11-18

  11: from google.adk import Runner, Workflow
  12: from google.adk.events import Event
  13: from google.adk.sessions.in_memory_session_service import InMemorySessionService
  14: from google.adk.workflow import node
  15: from google.genai import types
  16: 
  17: from app.agents.cache_policy import cache_policy_metadata
  18: from app.agents.pipeline_errors import PipelineError, RetrievalError, ServiceUnavailableError

# backend/app/agents/adk_workflow.py:476-514

 476: @node(name="planner")
 477: async def planner_node(ctx, node_input) -> dict[str, Any]:
 478:     """LLM agent: choose moral retrieval concepts and tone from the optimized dilemma."""
 479:     payload = node_input if isinstance(node_input, dict) else {}
 480:     state = ctx.state
 481:     dilemma = payload.get("dilemma") or state.get("dilemma") or _content_text(node_input)
 482:     optimized_query = payload.get("compressedQuery") or payload.get("optimizedQuery")
 483: 

 "The agent system is central to the product: each step has a bounded role, and the final answer only appears after retrieval and audit pass the product contract."

## 2. MCP Server

scripture retrieval is isolated behind a local stdio MCP server. The API workflow does not expose arbitrary tools; the client allowlists exactly the retrieval operations the agent is permitted to call.

In [22]:
show("backend/app/mcp/milvus_retrieval_server.py", 12, 21)
show("backend/app/mcp/milvus_retrieval_server.py", 64, 98)
show("backend/app/mcp/client.py", 21, 38)
show("backend/app/mcp/client.py", 121, 147)
show("backend/app/mcp/client.py", 178, 207)


# backend/app/mcp/milvus_retrieval_server.py:12-21

  12: from mcp.server.fastmcp import FastMCP
  13: 
  14: from app.config import get_settings
  15: from app.memory.milvus_store import MilvusStore
  16: from app.retrieval.corpus import expand_graph, get_corpus, load_scriptures_json
  17: from app.retrieval.hybrid_search import rerank_candidates
  18: 
  19: logger = logging.getLogger(__name__)
  20: 
  21: mcp = FastMCP("anayaa-milvus-retrieval")

# backend/app/mcp/milvus_retrieval_server.py:64-98

  64: @mcp.tool()
  65: def milvus_hybrid_search(
  66:     query: str,
  67:     keywords: list[str] | None = None,
  68:     limit: int = 20,
  69: ) -> dict:
  70:     """Search scripture verses using Milvus HNSW+BM25 hybrid search."""
  71:     store = _get_store()
  72:     keywords = keywords or []
  73:     results = store.hybrid_search(query, keywords, limit=limit)
  74:     return {"results": results, "source": "milvus", "count": len(results)}
  75: 
  76: 
  77: @mcp.tool()
  7

"The retrieval tool boundary is explicit: the agent can search, graph-expand, and rerank scripture, but it cannot call arbitrary local tools."

## 3. Security Features

user input passes through sanitizer, firewall, sensitive-name detection, and PII scrubbing before the workflow sees it. Output is audited for faithfulness, citation grounding, relevance, harmlessness, and privacy before it is returned.

In [12]:
show("backend/app/api/routes/query.py", 77, 109)
show("backend/app/security/firewall.py", 4, 17)
show("backend/app/security/firewall.py", 27, 44)
show("backend/app/security/privacy_scrubber.py", 6, 18)
show("backend/app/observability/g_eval_judge.py", 14, 35)


# backend/app/api/routes/query.py:77-109

  77: @router.post("/query")
  78: async def query(body: QueryBody, request: Request, user=Depends(require_auth)):
  79:     settings = get_settings()
  80:     pg = request.app.state.pg
  81:     redis = request.app.state.redis
  82:     session_mgr = request.app.state.session_mgr
  83: 
  84:     health = await check_health(pg, redis, getattr(request.app.state, "milvus_status", None))
  85:     if not health.get("corpusReady"):
  86:         raise HTTPException(status_code=503, detail="Corpus not loaded.")
  87: 
  88:     session_id = user.get("session_id")
  89:     if session_id and not await session_mgr.check_rate_limit(session_id, settings.rate_limit_per_minute):
  90:         raise HTTPException(status_code=429, detail="Rate limit exceeded.")
  91: 
  92:     # Security gates run before any agent, model, retrieval tool, or persistence path sees the query.
  93:     raw_query = sanitize_query(body.query)
  94:     sensitive_names = dete

## 4. Deployability

 Anayaa is packaged as a local-first beta with one public command path. The CLI owns setup, service startup, diagnostics, release checks, stopping, and cleanup.

In [13]:
show("scripts/install-anayaa.sh", 24, 46)
show("scripts/anayaa", 41, 67)
show("scripts/anayaa", 270, 310)
show("scripts/anayaa", 469, 498)
show("scripts/anayaa", 536, 545)


# scripts/install-anayaa.sh:24-46

  24: usage() {
  25:   cat <<'EOF'
  26: Usage:
  27:   ./scripts/install-anayaa.sh [options]
  28:   curl -sSL <release-install-url> | bash
  29: 
  30: Installs the `anayaa` command. From a checkout, this links to scripts/anayaa.
  31: When run through curl, it downloads a release archive into ~/.anayaa/Anayaa.AI.
  32: Rerunning a release install updates Anayaa code while preserving local runtime state.
  33: 
  34: Options:
  35:   --bin-dir DIR     Install the command link in DIR instead of ~/.local/bin
  36:   --install-dir DIR Install downloaded release files in DIR instead of ~/.anayaa/Anayaa.AI
  37:   --release-url URL Download this release archive when not running from a checkout
  38:   --replace         Replace an existing downloaded install directory without preserving local runtime state
  39:   --no-system-deps  Do not install/start system dependencies; only check them
  40:   --check-only      Check prerequisites and print next step

## 5. Antigravity

<video controls width="720">
  <source src="anayaa-antigravity.mov" type="video/mov">
</video>

## 6. Agent Skills / CLI

The user-facing `anayaa` command packages the product workflow into practical skills: setup, serve, doctor, release-check, stop, and clean. The internal agent workflow also exposes product-visible agent roles through latency and trace metadata.

In [14]:
show("scripts/anayaa", 45, 67)
show("backend/app/agents/adk_workflow.py", 158, 181)
show("backend/app/agents/adk_workflow.py", 417, 424)
show("backend/app/observability/latency.py", 33, 91)


# scripts/anayaa:45-67

  45: Commands:
  46:   setup             One-time online setup: local services, deps, models, embeddings, retrieval seed, frontend build
  47:   serve             Start local Anayaa at http://127.0.0.1:8000
  48:   doctor            Check local runtime prerequisites and cached assets
  49:   release-check     Run doctor plus compile/tests/frontend build checks
  50:   stop              Stop Anayaa app processes without deleting local assets
  51:   clean             Run the existing cleanup script; pass through cleanup options
  52:   help              Show this help
  53: 
  54: Serve options:
  55:   --offline         Force OFFLINE_MODE=true for cached local runtime
  56:   --online          Force OFFLINE_MODE=false; may install/pull if assets are missing
  57:   --reload          Start FastAPI with reload for development
  58: 
  59: Examples:
  60:   ${CLI_CMD} setup
  61:   ${CLI_CMD} serve
  62:   ${CLI_CMD} serve --online
  63:   ${CLI_CMD} doctor
  64:

 "The CLI is the practical skill surface for beta users, while the ADK workflow is the reasoning skill surface inside the product."